# STREAM velocity embedding stream plots

This notebook loads the trained standard CFM, FiLM STREAM, and cross-attention STREAM checkpoints, predicts velocities for a sampled set of cells, and saves one `scvelo.pl.velocity_embedding_stream` plot per model variant.

The notebook expects the STREAM outputs to exist under `outputs/stream/` and the full-data UMAP coordinates from the EDA workflow under `outputs/jax_adata_eda/`.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).resolve()
SRC_DIR = PROJECT_ROOT / "src"
USER_SITE = Path.home() / ".local" / "lib" / "python3.11" / "site-packages"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
if USER_SITE.exists() and str(USER_SITE) not in sys.path:
    sys.path.append(str(USER_SITE))

import gc
import torch
from stream_model.config import StreamConfig
from stream_model.scvelo_plotting import (
    add_velocity_layer,
    order_days_for_plot,
    plot_velocity_stream,
    predict_variant_velocity,
    sample_model_adata,
)

CONFIG_PATH = PROJECT_ROOT / "configs" / "stream_mouse_dev.yaml"
cfg = StreamConfig.from_yaml(CONFIG_PATH)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {device}")

Project root: /gpfs/commons/home/daknowles/projects/stream
Device: cuda


In [2]:
N_PLOT_CELLS = int(os.environ.get("STREAM_SCVELO_CELLS", "2500"))
PREDICT_BATCH_SIZE = int(os.environ.get("STREAM_SCVELO_BATCH", "32"))
SEED = int(os.environ.get("STREAM_SCVELO_SEED", "1337"))
OUT_DIR = cfg.out_dir / "scvelo_stream"
OUT_DIR.mkdir(parents=True, exist_ok=True)

adata = sample_model_adata(cfg, n_cells=N_PLOT_CELLS, seed=SEED)
adata = order_days_for_plot(adata)
print(adata)
print(adata.obs[["day", "source_file"]].head())

AnnData object with n_obs × n_vars = 2500 × 1984
    obs: 'cell_id', 'day', 'embryo_id', 'experimental_batch', 'source_file', 'global_idx'
    var: 'gene_short_name', 'gene_type', 'chr'
    obsm: 'X_umap'
                                    day               source_file
cell_id                                                          
run_4_P2-01G.AAACCATAGTTTAGGACCGG  E8.5  adata_JAX_dataset_1.h5ad
run_4_P2-04E.CCATCGTCTAATGCCGCTT   E8.5  adata_JAX_dataset_1.h5ad
run_4_P2-07C.CTAACGACTTTCCAACCGC   E8.5  adata_JAX_dataset_1.h5ad
run_4_P2-11C.ATAGTTGACTAGTAACGGTC  E8.5  adata_JAX_dataset_1.h5ad
run_4_PB-05H.ATAGTTGACTGAGCATATGG  E8.5  adata_JAX_dataset_1.h5ad


In [3]:
variants = ["standard_cfm", "film", "cross_attention"]
plot_paths = {}

for variant in variants:
    print(f"Predicting velocities for {variant}")
    velocity = predict_variant_velocity(
        adata,
        cfg,
        variant=variant,
        device=device,
        batch_size=PREDICT_BATCH_SIZE,
    )
    variant_adata = add_velocity_layer(adata, velocity, vkey="velocity")
    output_path = OUT_DIR / f"velocity_embedding_stream_{variant}.png"
    print(f"Plotting {output_path}")
    plot_velocity_stream(
        variant_adata,
        output_path=output_path,
        vkey="velocity",
        color="day",
        n_neighbors=30,
    )
    plot_paths[variant] = output_path

plot_paths

Predicting velocities for standard_cfm


Plotting /gpfs/commons/home/daknowles/projects/stream/outputs/stream/scvelo_stream/velocity_embedding_stream_standard_cfm.png


computing neighbors


    finished (0:00:08)
computing velocity graph (using 1/48 cores)


  0%|          | 0/2500 [00:00<?, ?cells/s]

...

Plotting /gpfs/commons/home/daknowles/projects/stream/outputs/stream/scvelo_stream/velocity_embedding_stream_cross_attention.png
computing neighbors


    finished (0:00:00)
computing velocity graph (using 1/48 cores)


  0%|          | 0/2500 [00:00<?, ?cells/s]

    finished (0:00:02)
computing velocity embedding
    finished (0:00:00)


{'standard_cfm': PosixPath('/gpfs/commons/home/daknowles/projects/stream/outputs/stream/scvelo_stream/velocity_embedding_stream_standard_cfm.png'),
 'film': PosixPath('/gpfs/commons/home/daknowles/projects/stream/outputs/stream/scvelo_stream/velocity_embedding_stream_film.png'),
 'cross_attention': PosixPath('/gpfs/commons/home/daknowles/projects/stream/outputs/stream/scvelo_stream/velocity_embedding_stream_cross_attention.png')}